임포트

In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path

경로 설정

In [3]:
cwd = Path(os.getcwd()).resolve()

def find_project_root(start: Path, max_up: int = 6) -> Path:
    cur = start
    for _ in range(max_up):
        if (cur / "data" / "processed").exists():
            return cur
        cur = cur.parent
    return start.parent

PROJECT_ROOT = find_project_root(cwd)
PROCESSED = PROJECT_ROOT / "data" / "processed"

PITCH_PATH = PROCESSED / "pitch_umap_cluster.parquet"
SUMMARY_PATH = PROCESSED / "pitcher_cluster_summary.csv"

OUT_PITCHER = PROCESSED / "pitcher_profiles.parquet"
OUT_BATTER  = PROCESSED / "batter_profiles.parquet"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PITCH_PATH:", PITCH_PATH, "exists?", PITCH_PATH.exists())
print("SUMMARY_PATH:", SUMMARY_PATH, "exists?", SUMMARY_PATH.exists())
print("OUT_PITCHER:", OUT_PITCHER)
print("OUT_BATTER:", OUT_BATTER)

PROJECT_ROOT: C:\Users\Dalab-server\Documents\kangmin\Pitcheezy\data_preprocess
PITCH_PATH: C:\Users\Dalab-server\Documents\kangmin\Pitcheezy\data_preprocess\data\processed\pitch_umap_cluster.parquet exists? True
SUMMARY_PATH: C:\Users\Dalab-server\Documents\kangmin\Pitcheezy\data_preprocess\data\processed\pitcher_cluster_summary.csv exists? True
OUT_PITCHER: C:\Users\Dalab-server\Documents\kangmin\Pitcheezy\data_preprocess\data\processed\pitcher_profiles.parquet
OUT_BATTER: C:\Users\Dalab-server\Documents\kangmin\Pitcheezy\data_preprocess\data\processed\batter_profiles.parquet


데이터 로드

In [4]:
dfp = pd.read_parquet(PITCH_PATH)
summary = pd.read_csv(SUMMARY_PATH)

print("pitch-level:", dfp.shape)
print("summary:", summary.shape)

dfp.head(3)

pitch-level: (719509, 40)
summary: (872, 20)


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,batter,pitcher,events,description,zone,...,bat_score_diff,arm_angle,description_group,events_group,base_state,count,umap_x,umap_y,pitch_cluster_local,pitch_cluster_id
0,FF,2025-03-28,97.8,-2.47,5.80,656811,592332,field_out,hit_into_play,11.0,...,-3,38.4,inplay,out,0,1-2,15.180676,1.919086,0,592332_0
1,SL,2025-03-28,85.8,-2.45,5.63,656811,592332,NaN,foul,5.0,...,-3,35.5,foul,other,0,1-2,-2.767725,13.609803,1,592332_1
2,FF,2025-03-28,95.8,-2.33,5.89,656811,592332,NaN,foul,1.0,...,-3,40.6,foul,other,0,1-1,14.007529,1.559027,0,592332_0


기본 컬럼 체크

In [5]:
required = ["pitcher", "batter", "pitch_type", "pitch_cluster_local", "pitch_cluster_id"]
missing = [c for c in required if c not in dfp.columns]
if missing:
    raise KeyError(f"Missing required columns in pitch_umap_cluster: {missing}")

# 숫자형 정리
dfp["pitcher"] = pd.to_numeric(dfp["pitcher"], errors="coerce").astype(int)
dfp["batter"]  = pd.to_numeric(dfp["batter"], errors="coerce").astype(int)

A. Pitcher Profiles (전 투수)

어떤 투수가 클러스터 기반이고, 어떤 투수가 fallback인지 결정

In [6]:
# summary의 did_cluster가 1이면 클러스터 기반 프로필
# (summary에 해당 투수가 없는 경우도 대비: 기본은 fallback)
cluster_ok = set(summary.loc[summary["did_cluster"] == 1, "pitcher"].astype(int).tolist())

all_pitchers = sorted(dfp["pitcher"].unique().tolist())
print("unique pitchers in pitch-level:", len(all_pitchers))
print("cluster_ok pitchers:", len(cluster_ok))

unique pitchers in pitch-level: 799
cluster_ok pitchers: 707


클러스터 기반 프로필 생성 (did_cluster=1 투수)

프로필 구성(기본):
클러스터 비율(=pitch mix)
클러스터별 평균 feature(구속/스핀/무브먼트)

In [7]:
PITCH_FEATURES = [
    "release_speed", "release_spin_rate", "pfx_x", "pfx_z",
    "release_pos_x", "release_pos_z", "release_extension", "arm_angle"
]

# 실제 존재하는 feature만 사용
PITCH_FEATURES = [c for c in PITCH_FEATURES if c in dfp.columns]
print("features used:", PITCH_FEATURES)

df_cluster = dfp[dfp["pitcher"].isin(cluster_ok)].copy()

# (1) 클러스터 비율
cluster_counts = (
    df_cluster.groupby(["pitcher", "pitch_cluster_id"])
    .size()
    .rename("n")
    .reset_index()
)

totals = cluster_counts.groupby("pitcher")["n"].sum().rename("n_total").reset_index()
cluster_counts = cluster_counts.merge(totals, on="pitcher", how="left")
cluster_counts["ratio"] = cluster_counts["n"] / cluster_counts["n_total"]

mix_wide = cluster_counts.pivot(index="pitcher", columns="pitch_cluster_id", values="ratio").fillna(0.0)
mix_wide.columns = [f"mix_{c}" for c in mix_wide.columns]  # 컬럼명 정리

# (2) 클러스터별 평균 feature -> 너무 컬럼이 많아지므로 "가중 평균(투수 전체 기준)"도 같이 제공
# a) 클러스터별 평균(롱 형태)
cluster_feat_mean = (
    df_cluster.groupby(["pitcher", "pitch_cluster_id"])[PITCH_FEATURES]
    .mean()
    .reset_index()
)

# b) 투수 전체 가중 평균 feature (클러스터 mix로 가중)
#   투수별로 feature의 "클러스터 가중 평균"을 구해 1개 벡터로 저장
tmp = cluster_feat_mean.merge(cluster_counts[["pitcher","pitch_cluster_id","ratio"]], on=["pitcher","pitch_cluster_id"], how="left")
for c in PITCH_FEATURES:
    tmp[c] = tmp[c] * tmp["ratio"]

pitcher_feat = tmp.groupby("pitcher")[PITCH_FEATURES].sum()
pitcher_feat.columns = [f"feat_{c}_wavg" for c in pitcher_feat.columns]

# 합치기: mix + weighted feature
pitcher_profile_cluster = mix_wide.join(pitcher_feat, how="outer").fillna(0.0)
pitcher_profile_cluster["profile_source"] = "cluster"
pitcher_profile_cluster.head(3)

features used: ['release_speed', 'release_spin_rate', 'pfx_x', 'pfx_z', 'release_pos_x', 'release_pos_z', 'release_extension', 'arm_angle']


,mix_434378_-1,mix_434378_0,mix_434378_1,mix_434378_2,mix_434378_3,mix_434378_4,mix_434378_5,mix_434378_6,mix_445276_0,mix_445276_1,...,mix_829272_3,feat_release_speed_wavg,feat_release_spin_rate_wavg,feat_pfx_x_wavg,feat_pfx_z_wavg,feat_release_pos_x_wavg,feat_release_pos_z_wavg,feat_release_extension_wavg,feat_arm_angle_wavg,profile_source
pitcher,,,,,,,,,,,,,,,,,,,,,
434378,0.005368,0.084739,0.266104,0.191718,0.135736,0.097776,0.100077,0.118482,0.000000,0.000000,...,0.0,88.247048,2455.433282,-0.184214,0.722423,-1.833923,7.100084,6.013305,55.150000,cluster
445276,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.089965,0.094579,...,0.0,91.931949,2566.124567,0.172710,1.409239,-2.167993,6.340911,6.896886,52.562168,cluster
445926,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,86.685965,2086.801170,-0.651930,0.574386,-1.526433,5.010351,6.244444,21.139766,cluster


fallback 프로필 생성 (did_cluster=0 투수: pitch_type 기반) 

In [8]:
fallback_pitchers = [p for p in all_pitchers if p not in cluster_ok]
df_fallback = dfp[dfp["pitcher"].isin(fallback_pitchers)].copy()

# pitch_type 비율
pt_counts = (
    df_fallback.groupby(["pitcher", "pitch_type"])
    .size()
    .rename("n")
    .reset_index()
)

pt_totals = pt_counts.groupby("pitcher")["n"].sum().rename("n_total").reset_index()
pt_counts = pt_counts.merge(pt_totals, on="pitcher", how="left")
pt_counts["ratio"] = pt_counts["n"] / pt_counts["n_total"]

pt_wide = pt_counts.pivot(index="pitcher", columns="pitch_type", values="ratio").fillna(0.0)
pt_wide.columns = [f"mix_pitchtype_{c}" for c in pt_wide.columns]

# pitch_type별 feature 평균을 "투수 전체 기준 가중 평균"으로만 저장(컬럼 폭발 방지)
if len(PITCH_FEATURES) > 0:
    pt_feat = (
        df_fallback.groupby(["pitcher", "pitch_type"])[PITCH_FEATURES]
        .mean()
        .reset_index()
        .merge(pt_counts[["pitcher","pitch_type","ratio"]], on=["pitcher","pitch_type"], how="left")
    )
    for c in PITCH_FEATURES:
        pt_feat[c] = pt_feat[c] * pt_feat["ratio"]

    pitcher_feat_fb = pt_feat.groupby("pitcher")[PITCH_FEATURES].sum()
    pitcher_feat_fb.columns = [f"feat_{c}_wavg" for c in pitcher_feat_fb.columns]
else:
    pitcher_feat_fb = pd.DataFrame(index=pt_wide.index)

pitcher_profile_fallback = pt_wide.join(pitcher_feat_fb, how="outer").fillna(0.0)
pitcher_profile_fallback["profile_source"] = "pitch_type_fallback"

pitcher_profile_fallback.head(3)

,mix_pitchtype_CH,mix_pitchtype_CS,mix_pitchtype_CU,mix_pitchtype_EP,mix_pitchtype_FA,mix_pitchtype_FC,mix_pitchtype_FF,mix_pitchtype_FS,mix_pitchtype_KC,mix_pitchtype_KN,...,mix_pitchtype_SV,feat_release_speed_wavg,feat_release_spin_rate_wavg,feat_pfx_x_wavg,feat_pfx_z_wavg,feat_release_pos_x_wavg,feat_release_pos_z_wavg,feat_release_extension_wavg,feat_arm_angle_wavg,profile_source
pitcher,,,,,,,,,,,,,,,,,,,,,
476594,0.050847,0.0,0.0,0.00000,0.00000,0.440678,0.101695,0.0,0.0,0.0,...,0.0,89.964407,2234.864407,-0.542034,0.342712,-3.278983,4.553051,6.305085,5.461017,pitch_type_fallback
493603,0.000000,0.0,0.0,0.00000,0.00000,0.071429,0.238095,0.0,0.0,0.0,...,0.0,87.602381,2424.833333,-0.219048,0.645000,-2.660952,5.388095,7.028571,24.716667,pitch_type_fallback
500743,0.000000,0.0,0.0,0.84058,0.15942,0.000000,0.000000,0.0,0.0,0.0,...,0.0,47.881159,1292.623188,-0.201304,1.428406,-1.305942,6.620290,4.618841,42.597101,pitch_type_fallback


전 투수 프로필 통합 + 최소 컬럼 정리

In [9]:
pitcher_profiles = pd.concat([pitcher_profile_cluster, pitcher_profile_fallback], axis=0)

# pitcher를 인덱스 -> 컬럼으로
pitcher_profiles = pitcher_profiles.reset_index().rename(columns={"index":"pitcher"})

# 요약 정보 붙이기(투구 수, did_cluster 등)
summary_small = summary.copy()
summary_small["pitcher"] = summary_small["pitcher"].astype(int)

pitcher_profiles = pitcher_profiles.merge(
    summary_small[["pitcher","n_pitches","did_umap","did_cluster","n_clusters","noise_ratio"]],
    on="pitcher",
    how="left"
)

# 결측 채우기(혹시 summary에 없는 투수)
for c in ["did_umap","did_cluster","n_clusters"]:
    if c in pitcher_profiles.columns:
        pitcher_profiles[c] = pitcher_profiles[c].fillna(0).astype(int)

if "n_pitches" in pitcher_profiles.columns:
    pitcher_profiles["n_pitches"] = pitcher_profiles["n_pitches"].fillna(
        dfp.groupby("pitcher").size()
    ).astype(int)

print("pitcher_profiles:", pitcher_profiles.shape)
pitcher_profiles.head(5)

pitcher_profiles: (799, 2379)


,pitcher,mix_434378_-1,mix_434378_0,mix_434378_1,mix_434378_2,mix_434378_3,mix_434378_4,mix_434378_5,mix_434378_6,mix_445276_0,...,mix_pitchtype_KN,mix_pitchtype_SI,mix_pitchtype_SL,mix_pitchtype_ST,mix_pitchtype_SV,n_pitches,did_umap,did_cluster,n_clusters,noise_ratio
0,434378,0.005368,0.084739,0.266104,0.191718,0.135736,0.097776,0.100077,0.118482,0.000000,...,NaN,NaN,NaN,NaN,NaN,2608,1,1,7,0.005368
1,445276,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.089965,...,NaN,NaN,NaN,NaN,NaN,867,1,1,3,0.000000
2,445926,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,171,1,1,2,0.000000
3,448179,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,171,1,1,2,0.000000
4,450203,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,2512,1,1,4,0.004379


pitcher_profiles 저장

In [10]:
pitcher_profiles.to_parquet(OUT_PITCHER, index=False)
print("saved:", OUT_PITCHER)

saved: C:\Users\Dalab-server\Documents\kangmin\Pitcheezy\data_preprocess\data\processed\pitcher_profiles.parquet


B.Batter Profiles (전 타자)

batter profile 생성

In [11]:
# 가능한 컬럼만 사용(01에서 events_group/description_group을 만들었다면 포함)
batter_cols = ["batter", "stand", "launch_speed", "launch_angle"]
extra_dist_cols = []
for c in ["events_group", "description_group"]:
    if c in dfp.columns:
        extra_dist_cols.append(c)

dfb = dfp.copy()

# 기본 numeric 요약
batter_num = dfb.groupby("batter")[["launch_speed","launch_angle"]].mean()
batter_num.columns = ["batter_avg_launch_speed", "batter_avg_launch_angle"]

# stand는 가장 빈도 높은 값으로
if "stand" in dfb.columns:
    batter_stand = dfb.groupby("batter")["stand"].agg(lambda x: x.value_counts().index[0])
    batter_stand = batter_stand.to_frame("batter_stand_mode")
else:
    batter_stand = pd.DataFrame(index=batter_num.index, columns=["batter_stand_mode"])

# events/description 분포(가능할 때)
dist_frames = []
for c in extra_dist_cols:
    cnt = (
        dfb.groupby(["batter", c]).size().rename("n").reset_index()
    )
    tot = cnt.groupby("batter")["n"].sum().rename("n_total").reset_index()
    cnt = cnt.merge(tot, on="batter", how="left")
    cnt["ratio"] = cnt["n"] / cnt["n_total"]
    wide = cnt.pivot(index="batter", columns=c, values="ratio").fillna(0.0)
    wide.columns = [f"batter_{c}_{v}" for v in wide.columns]
    dist_frames.append(wide)

batter_profiles = batter_num.join(batter_stand, how="outer")
for wide in dist_frames:
    batter_profiles = batter_profiles.join(wide, how="outer")

batter_profiles = batter_profiles.fillna(0.0).reset_index()

# 타자 투구 수
batter_profiles["n_pitches_seen"] = dfb.groupby("batter").size().values

print("batter_profiles:", batter_profiles.shape)
batter_profiles.head(5)

batter_profiles: (674, 15)


,batter,batter_avg_launch_speed,batter_avg_launch_angle,batter_stand_mode,batter_events_group_hit,batter_events_group_other,batter_events_group_out,batter_events_group_reached,batter_events_group_walk,batter_description_group_ball,batter_description_group_foul,batter_description_group_inplay,batter_description_group_other,batter_description_group_strike,n_pitches_seen
0,455117,20.072935,3.780316,R,0.052724,0.731107,0.200351,0.001757,0.014060,0.319859,0.117750,0.168717,0.000000,0.393673,569
1,456781,30.189941,6.778107,R,0.057692,0.751479,0.173077,0.001479,0.016272,0.304734,0.210059,0.180473,0.004438,0.300296,676
2,457705,26.005407,5.364611,R,0.050045,0.763181,0.159071,0.000447,0.027256,0.384718,0.185433,0.160411,0.000894,0.268543,2238
3,457759,30.765923,9.070632,R,0.047088,0.765799,0.163569,0.001239,0.022305,0.370508,0.229244,0.171004,0.001239,0.228005,807
4,467793,26.213061,5.653571,L,0.046429,0.762245,0.163265,0.001020,0.027041,0.392857,0.181122,0.167347,0.001531,0.257143,1960


batter_profiles 저장

In [14]:
batter_profiles.to_parquet(OUT_BATTER, index=False)
print("saved:", OUT_BATTER)

saved: C:\Users\Dalab-server\Documents\kangmin\Pitcheezy\data_preprocess\data\processed\batter_profiles.parquet


프로필 파일 로드 확인

In [15]:
pp = pd.read_parquet(OUT_PITCHER)
bp = pd.read_parquet(OUT_BATTER)

print("pitcher_profiles:", pp.shape)
print("batter_profiles:", bp.shape)

pp.head(3), bp.head(3)

pitcher_profiles: (799, 2379)
batter_profiles: (674, 15)


(   pitcher  mix_434378_-1  mix_434378_0  mix_434378_1  mix_434378_2  \
 0   434378       0.005368      0.084739      0.266104      0.191718   
 1   445276       0.000000      0.000000      0.000000      0.000000   
 2   445926       0.000000      0.000000      0.000000      0.000000   
 
    mix_434378_3  mix_434378_4  mix_434378_5  mix_434378_6  mix_445276_0  ...  \
 0      0.135736      0.097776      0.100077      0.118482      0.000000  ...   
 1      0.000000      0.000000      0.000000      0.000000      0.089965  ...   
 2      0.000000      0.000000      0.000000      0.000000      0.000000  ...   
 
    mix_pitchtype_KN  mix_pitchtype_SI  mix_pitchtype_SL  mix_pitchtype_ST  \
 0               NaN               NaN               NaN               NaN   
 1               NaN               NaN               NaN               NaN   
 2               NaN               NaN               NaN               NaN   
 
    mix_pitchtype_SV  n_pitches  did_umap  did_cluster  n_clusters  no